# Fine-tuning walkthrough

This notebook drives the `llm_finetune` package end to end. The loop is always the same: pick a base model, format the data, attach a LoRA adapter, train, then measure. LoRA freezes the base weights and learns a few small matrices, so one run touches a tiny fraction of the parameters.

We run a tiny offline GPT-2 here so the whole thing finishes in seconds and needs no network. That model is for the mechanics, not the quality. It cannot write fluent English no matter how well it trains. The lesson is the before and after perplexity on the training data, which shows the LoRA update learned something. The last cell shows how to swap in a real model for a run that produces sensible text.

In [ ]:
import os

from llm_finetune import data, evaluate, model, train
from llm_finetune.config import TrainConfig

from common.dl import get_device, set_seed

set_seed(42)
device = get_device()
print(f"device: {device}")

## 1. Config

`TrainConfig` holds every knob the data, model and train modules read. We keep it small: `base_model="tiny"` builds the offline GPT-2, a short sequence length, a few epochs so perplexity visibly drops on the small corpus, and a LoRA rank of 8.

In [ ]:
config = TrainConfig(
    base_model="tiny",
    mode="instruct",
    max_seq_length=64,
    num_train_epochs=8,
    per_device_train_batch_size=2,
    lora_rank=8,
    lora_alpha=16,
    learning_rate=5e-3,
)
print(config)

## 2. Data

`demo_corpus()` is a small offline instruct set about the fictional island of Marru. `build_dataset` formats each record, tokenises it, and makes a deterministic train/validation split. We need the tokenizer first, so we load the model in the next cell and build the dataset there. For now, look at one record.

In [ ]:
records = data.demo_corpus()
print(f"rows: {len(records)}")
print(records[0])

## 3. Model and LoRA

`load_base("tiny")` returns a two-layer GPT-2 and a matching tokenizer, both built offline. `add_lora` wraps the model in a LoRA adapter on the attention projection. The trainable count is a small slice of the total: only the adapter trains, the base weights stay frozen.

In [ ]:
base_model, tokenizer = model.load_base("tiny")
lora_model = model.add_lora(base_model, config)

trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in lora_model.parameters())
print(f"trainable params: {trainable:,}")
print(f"total params:     {total:,}")
print(f"trainable share:  {100 * trainable / total:.2f}%")

In [ ]:
dataset = data.build_dataset(records, tokenizer, config)
print(f"train rows:      {len(dataset['train'])}")
print(f"validation rows: {len(dataset['validation'])}")

## 4. Baseline

Before training, measure perplexity on a few training texts and generate a sample. Perplexity is the exponential of the mean per-token cross-entropy, so lower is better. The generation is nonsense: the base GPT-2 has random weights and a tiny vocabulary, so it has nothing sensible to say yet.

In [ ]:
eval_texts = [data.format_instruct(r, tokenizer) for r in records[:4]]
prompt = "Instruction:\nWhat is the capital of Marru?\n\nResponse:\n"

ppl_before = evaluate.perplexity(lora_model, tokenizer, eval_texts, device)
sample_before = evaluate.generate(lora_model, tokenizer, prompt, max_new_tokens=20)
print(f"perplexity before: {ppl_before:.2f}")
print(f"sample before: {sample_before!r}")

## 5. Train

`train.train` drives `transformers.Trainer` over the dataset and returns a small metrics dict. For the tiny model it forces CPU, which is fast and deterministic. The training loss should fall across the run.

In [ ]:
result = train.train(lora_model, tokenizer, dataset, config)
print(f"train_loss: {result['train_loss']:.4f}")
print(f"steps:      {result['steps']}")

## 6. After

Measure perplexity again on the same texts, with the same tokenizer, and generate once more. Perplexity should drop: the adapter has pulled probability towards the training sequences. The sample is still gibberish. A two-layer model cannot form real words, so the falling perplexity, not the text, is the evidence that the LoRA update learned something.

In [ ]:
ppl_after = evaluate.perplexity(lora_model, tokenizer, eval_texts, device)
sample_after = evaluate.generate(lora_model, tokenizer, prompt, max_new_tokens=20)
print(f"perplexity before: {ppl_before:.2f}")
print(f"perplexity after:  {ppl_after:.2f}")
print(f"drop:              {ppl_before - ppl_after:.2f}")
print(f"sample after: {sample_after!r}")

## 7. The real-run swap

The same pipeline runs on a real base model by changing one setting. Set the `LLM_BASE_MODEL` environment variable to a Hugging Face model id, for example `Qwen/Qwen2.5-0.5B-Instruct`, and run again. That is roughly a 1 GB download and a few minutes to train a small LoRA on MPS or CPU. With a real model the generations become coherent and the perplexity figures mean something across runs. The exact numbers depend on your hardware and corpus, so measure them on your own run.

The cell below is gated on the environment variable, so a default offline run skips it. The `try/except` catches a missing model or package and prints a one-line note instead of failing.

In [ ]:
real_id = os.environ.get("LLM_BASE_MODEL")
if real_id:
    try:
        real_config = TrainConfig(
            base_model=real_id,
            mode="instruct",
            max_seq_length=128,
            num_train_epochs=3,
            lora_rank=8,
        )
        real_model, real_tok = model.load_base(real_id)
        real_model = model.add_lora(real_model, real_config)
        real_ds = data.build_dataset(data.demo_corpus(), real_tok, real_config)
        real_result = train.train(real_model, real_tok, real_ds, real_config)
        print(f"real train_loss: {real_result['train_loss']:.4f}")
        print(evaluate.generate(real_model, real_tok, prompt, max_new_tokens=50))
    except (OSError, ImportError) as error:
        print(f"skipping real run: {error}")
else:
    print("LLM_BASE_MODEL not set; skipping the real-model run.")

## What breaks

A tiny model cannot produce fluent text, however low the loss goes. Real gains need a real base model and more data than twelve rows. Perplexity is only comparable with the same tokenizer, so never read a GPT-2 number against a Qwen number. The falling perplexity here proves the training loop and the adapter work, which is what this walkthrough set out to show.